**Home Exercise 1 on Machine Translation**
Implement a `sequence2sequence` to **translate English to Vietnamese**. In this exercise, we will sequentially practice the steps to build a machine learning system for the machine translation task using a `seq2seq` model. These steps *include downloading and preprocessing bilingual data, creating training data, building a `seq2seq` model with attention, visualizing attention data, and translating new sentences on real-world data*.

Data: [IWSLT'15 English-Vietnamese](https://www.kaggle.com/datasets/tuannguyenvananh/iwslt15-englishvietnamese) (Train set: train.en and train.vi||| Val set: tst2012.en and tst2012.vi ||| Test set: tst2013.en and tst2013.vi).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tqdm
import shutil, sys, zipfile
import random

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from datetime import datetime
import datetime

print(f"The last time this notebook was run is: {datetime.datetime.now().strftime('%H:%M:%S %d/%m/%')}")

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

The last time this notebook was run is: 20:32:33 28/11/%


In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("tuannguyenvananh/iwslt15-englishvietnamese")

print("Path to dataset files:", path)

100%|██████████| 10.3M/10.3M [00:01<00:00, 6.80MB/s]

Extracting files...


Path to dataset files: /home/dikhang/.cache/kagglehub/datasets/tuannguyenvananh/iwslt15-englishvietnamese/versions/1


In [10]:
# Helper_functions
def unzip(path, dest, delete=True):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Zip file does not exist: {path}")
    
    if not zipfile.is_zipfile(path):
        raise zipfile.BadZipFile(f"Not a valid zip file: {path}")
    
    with zipfile.ZipFile(path, 'r') as zip_ref:
        zip_ref.extractall(dest)
        print(f"Unzipped into: {dest}")
    if delete:
        os.remove(path)
    else:
        print(f"Do not remove zipfile.")
    
    return dest

def move_path(src_path: str, dest_dir: str):
    if not os.path.exists(src_path):
        raise FileNotFoundError(f"Invalid {src_path}")
    
    os.makedirs(dest_dir, exist_ok=True)
    
    dst_path = os.path.join(dest_dir, os.path.basename(src_path))
    if os.path.exists(dst_path):
        print(f"Destination {dest_dir} is in used.")
        if os.path.isdir(dst_path):
            shutil.rmtree(dst_path)
        else:
            os.remove(dst_path)

    try:
        new_path = shutil.move(src_path, dest_dir)
        return new_path
    except Exception as e:
        if os.path.isdir(src_path):
            shutil.copytree(src_path, dst_path)
            shutil.rmtree(src_path)
            return dst_path
        else:
            raise e

In [ ]:
!ls /home/dikhang/.cache/kagglehub/datasets/tuannguyenvananh/iwslt15-englishvietnamese/versions/1

"IWSLT'15 en-vi"


In [12]:
src_dir = "/home/dikhang/.cache/kagglehub/datasets/tuannguyenvananh/iwslt15-englishvietnamese/versions/1"
filename = "IWSLT'15 en-vi"

data_dir = "./data"
full_path = os.path.join(src_dir, filename)
file_path = move_path(full_path, data_dir)
file_path = os.path.join(data_dir, filename)
print("Moved to:", file_path)

Moved to: ./data/IWSLT'15 en-vi
